# Helios 03 — Analytical Core: terrain, COGs, STAC, and solar scoring

**Solar site-selection, step 3: the analytical layer.** Roof solar yield depends on
**slope** and **aspect** (south-facing, gently sloped wins). This notebook stages a
**USGS 3DEP** DEM for San Francisco, converts it to **Cloud-Optimized GeoTIFFs** with
`gbx_rst_cog_convert`, **catalogs the COGs into a queryable Delta table** — a
time-travel-friendly catalog of the analysis-ready elevation assets — derives
slope/aspect/hillshade, aggregates them into a per-**H3-cell** `solar_score` index
(composing with Databricks-native H3), and renders the hillshade as PMTiles.

> GeoBrix tiling and raster→grid aggregation are an on-ramp into Databricks-native
> spatial: the H3 `cellid` we produce is a standard native id you can join and render
> with native H3 functions.

> Runs on the **lightweight tier (Serverless)** by default.


![3DEP DEM to COG + STAC + hillshade PMTiles](https://raw.githubusercontent.com/databrickslabs/geobrix/main/resources/images/diagrams/helios/helios-03.png)

In [ ]:
%run ./config_nb

In [ ]:
# Flip to True to fully rebuild this notebook's tables / re-download / re-tile.
FORCE_REBUILD = False

## 1. Stage a USGS 3DEP DEM (notebook helper)

USGS 3DEP elevation is on the public AWS Open Data registry and discoverable via
Planetary Computer STAC (collection `3dep-seamless`). Like NAIP, 3DEP stays a
**notebook-local helper** — no module API. We stage one SF DEM tile to the Volume
(idempotent). For offline runs (set `GBX_HELIOS_OFFLINE=1`) we fall back to the
committed `srtm_n37w123.tif` elevation sample already staged in the sample-data Volume.

In [ ]:
import os
import shutil
from databricks.labs.gbx.sample import get_temp_dir

DEM_DIR = f"{HELIOS_DIR}/dem"
os.makedirs(DEM_DIR, exist_ok=True)   # Volume FUSE-safe; idempotent
DEM_PATH = f"{DEM_DIR}/sf_3dep.tif"

_SAMPLE_ROOT = os.environ.get(
    "GBX_SAMPLE_DATA_ROOT",
    "/Volumes/main/default/geobrix_samples/geobrix-examples",
)
SRTM_FALLBACK = f"{_SAMPLE_ROOT}/sf/elevation/srtm_n37w123.tif"
_OFFLINE = os.environ.get("GBX_HELIOS_OFFLINE", "").lower() in ("1", "true", "yes")


def stage_dem(dest=DEM_PATH, bbox=SF_CITY_BBOX):
    """Stage one SF 3DEP DEM tile via Planetary Computer STAC; fall back to the
    already-staged SRTM tile when offline (idempotent)."""
    if os.path.exists(dest) and not FORCE_REBUILD:
        print(f"... DEM already staged at {dest}")
        return dest
    if _OFFLINE:
        print(f"... GBX_HELIOS_OFFLINE=1: copying SRTM fallback -> {dest}")
        shutil.copy(SRTM_FALLBACK, dest)
        return dest
    try:
        import planetary_computer as pc
        import pystac_client
        import rasterio
        from rasterio.warp import transform_bounds
        from rasterio.windows import from_bounds as window_from_bounds

        cat = pystac_client.Client.open(
            "https://planetarycomputer.microsoft.com/api/stac/v1",
            modifier=pc.sign_inplace,
        )
        minx, miny, maxx, maxy = bbox
        item = next(
            cat.search(
                collections=["3dep-seamless"],
                bbox=[minx, miny, maxx, maxy],
                limit=1,
            ).items()
        )
        href = item.assets["data"].href
        tmp = get_temp_dir()
        local = tmp / "sf_3dep.tif"
        with rasterio.open(href) as src:
            # 3DEP tiles may be in a projected CRS — transform bbox CRS-aware.
            proj_bounds = transform_bounds("EPSG:4326", src.crs, minx, miny, maxx, maxy)
            win = window_from_bounds(*proj_bounds, transform=src.transform)
            data = src.read(window=win)
            profile = {
                **src.profile,
                "driver": "GTiff",
                "width": data.shape[2],
                "height": data.shape[1],
                "transform": src.window_transform(win),
            }
            with rasterio.open(local, "w", **profile) as dst:
                dst.write(data)
        shutil.copy(str(local), dest)   # FUSE-safe sequential copy
        print(f"... staged 3DEP -> {dest} ({os.path.getsize(dest):,} bytes)")
    except Exception as e:
        print(f"... 3DEP STAC unavailable ({type(e).__name__}: {e}); falling back to SRTM")
        shutil.copy(SRTM_FALLBACK, dest)
    return dest


stage_dem()
plot_file(DEM_PATH, fig_w=8, fig_h=6)

## 2. Convert the DEM to Cloud-Optimized GeoTIFF

`gbx_rst_cog_convert` rewrites the raster as a COG (internal tiling + overviews) so
downstream tools can do fast windowed and overview reads. We load the DEM bytes with the
`binaryFile` reader, build a typed tile via `rst_fromcontent`, convert, then extract the
COG bytes from the `raster` field of the tile struct and write them to the Volume.

In [ ]:
# Load: binaryFile -> rst_fromcontent -> tile struct.
# rst_fromcontent requires an explicit driver string (GTiff for .tif).
# Fan out by SOURCE raster: read the DEM *directory* (one row per source) and
# repartition by `path` so cog_convert + slope/aspect/hillshade + H3 binning +
# the XYZ pyramid all run in parallel across sources. A column key is required —
# number-only repartition(N) is AQE-coalesced to serial on Serverless. The demo
# stages one DEM (one partition); point DEM_DIR at many tiles and it fans out.
dem = (
    spark.read.format("binaryFile").option("pathGlobFilter", "*.tif").load(DEM_DIR)
         .repartition(64, F.col("path"))
         .select(rx.rst_fromcontent(F.col("content"), F.lit("GTiff")).alias("tile"))
)

# Convert to COG layout (DEFLATE compression, 512-px blocks, AVERAGE overviews).
# gbx_rst_cog_convert signature: rst_cog_convert(tile [, compression [, blocksize [, overview_resampling]]])
cog = dem.select(rx.rst_cog_convert("tile").alias("tile"))

# Write COG bytes to Volume.
# The tile struct has a 'raster' field containing the raw GeoTIFF bytes.
COG_DIR = f"{HELIOS_DIR}/cog"
os.makedirs(COG_DIR, exist_ok=True)
COG_PATH = f"{COG_DIR}/sf_dem_cog.tif"

if not os.path.exists(COG_PATH) or FORCE_REBUILD:
    # Write ONE representative source COG for the inline inspection below (limit(1)
    # avoids collecting every source to the driver at scale). The full `cog`
    # DataFrame stays multi-row so the terrain/H3/pyramid steps fan out.
    cog_row = cog.select(F.col("tile.raster").alias("raster")).limit(1).collect()[0]
    with open(COG_PATH, "wb") as f:
        f.write(bytes(cog_row["raster"]))
    print(f"... wrote COG {COG_PATH} ({os.path.getsize(COG_PATH):,} bytes)")
else:
    print(f"... COG already exists at {COG_PATH} (skip; FORCE_REBUILD=False)")

## 3. View the COG

`show_cog` (from `config_nb`) does a rasterio overview read and renders the elevation
surface with a contextily basemap.

In [ ]:
show_cog(COG_PATH)

## 4. Catalog the COGs into a Delta table

We register the COG(s) as items in a queryable Delta table — a re-runnable,
time-travel-friendly catalog of the analysis-ready elevation assets, keyed by their
Volume path. `StacClient` (instantiated in `config_nb`) is a STAC-reader for remote
catalogs (Planetary Computer, AWS, etc.); it does not expose a catalog-build API for
local files. We hand-build the rows instead and materialize them with `finalize_delta`
— the same idempotent pattern the EO series uses.

In [ ]:
import rasterio

with rasterio.open(COG_PATH) as src:
    b = src.bounds
    cog_meta = [(
        "sf_dem_cog",
        COG_PATH,
        str(src.crs),
        float(b.left),
        float(b.bottom),
        float(b.right),
        float(b.top),
        "3dep-seamless",
    )]

cog_df = spark.createDataFrame(
    cog_meta,
    "item_id string, source string, crs string, "
    "minx double, miny double, maxx double, maxy double, collection string",
)
sf_cog_catalog = finalize_delta(cog_df, "sf_cog_catalog")
display(sf_cog_catalog)

## 5. Derive slope, aspect, hillshade, and a solar score

Slope and aspect come straight from the DEM tile.

Registered names (confirmed against `registered_functions.txt`):
- `gbx_rst_slope` / `rx.rst_slope` — slope in degrees, auto-scaled from CRS (no manual `-s`).
- `gbx_rst_aspect` / `rx.rst_aspect` — aspect in degrees, GDAL convention (0=N, 90=E, 180=S, 270=W).
- `gbx_rst_hillshade` / `rx.rst_hillshade` — relief shading at default azimuth/altitude.

`solar_score` (defined in `config_nb`) favors gently sloped, south-facing terrain. It is
applied per H3 cell in step 5b, where slope and aspect are aggregated onto the H3 grid.

In [ ]:
# gbx_rst_slope / gbx_rst_aspect / gbx_rst_hillshade: tile-in, tile-out.
# Slope/hillshade auto-scale from CRS per GDAL 3.11 formula — no manual scale arg.
# Aspect is scale-invariant.
# Fans out per source (the load cell repartitioned by `path`). NOTE: with MANY
# adjacent source DEMs, slope/hillshade are computed independently per source, so
# gradients can seam at shared edges — mosaic first, or split with overlapping
# source tiles + trim, for seamless terrain across a large area.
terrain = cog.select(
    rx.rst_slope("tile").alias("slope_tile"),
    rx.rst_aspect("tile").alias("aspect_tile"),
    rx.rst_hillshade("tile").alias("hillshade_tile"),
)
sf_terrain = finalize_delta(terrain, "sf_terrain", do_display=True)
print("... slope / aspect / hillshade tiles materialized")

## 5b. Aggregate slope + aspect into a per-H3-cell solar-suitability index

GeoBrix raster→grid aggregation bins the slope and aspect rasters onto **H3 cells**
(`gbx_rst_h3_rastertogridavg`), emitting a standard H3 integer `cellid` per cell.

Function contract (confirmed against light-tier UDTF registration and tests):
- Signature: `gbx_rst_h3_rastertogridavg(tile, resolution)` — **UDTF** invoked via
  SQL `LATERAL`. Each output row is `(band INT, cellID LONG, measure DOUBLE)`.
- Use `LATERAL gbx_rst_h3_rastertogridavg(tile, resolution) t` and filter
  `WHERE t.band = 1` to select band 1. No `[0]` indexing or `LATERAL VIEW explode`.
- **Expects EPSG:4326 lon/lat.** SRTM is already 4326; 3DEP may be EPSG:4269 (NAD83).
  The code calls `rx.rst_transform(tile, F.lit(4326))` on the slope/aspect tiles before
  aggregation — this is a no-op when the CRS is already 4326 and corrects NAD83 3DEP
  (cm-scale offset at SF, but correct by contract).

Because that `cellid` is a native H3 id, the result composes directly with
**Databricks-native H3** functions — `h3_centeraswkb` / `h3_boundaryaswkb` for cell
geometry, `h3_hexring` for neighborhoods. We gate the native-H3 geometry step behind a
capability check: `h3_centeraswkb` requires Photon or Databricks SQL, which may not be
available in all environments (e.g., Docker or local Spark).


In [ ]:
H3_RES = 9   # H3 resolution 9 (~175 m cells) — appropriate for a city-scale DEM.

# Register the pre-derived terrain tiles as a temp view for SQL access.
# sf_terrain has columns: slope_tile, aspect_tile, hillshade_tile.
sf_terrain.createOrReplaceTempView("sf_terrain_tiles")

# gbx_rst_h3_rastertogridavg is a light-tier UDTF invoked via SQL LATERAL.
# Each output row: band INT (1-based), cellID LONG (H3 integer id), measure DOUBLE.
# Filter WHERE t.band = 1 to select the first (and only) band.
# No [0] indexing or LATERAL VIEW explode — those are the heavy-tier pattern.
#
# rst_transform(tile, 4326) normalises CRS to EPSG:4326 before aggregation.
# SRTM is already 4326 (no-op); 3DEP may be EPSG:4269 (NAD83) — corrected here.
slope_cells = spark.sql(f"""
    SELECT t.cellID AS cellid, t.measure AS avg_slope
    FROM sf_terrain_tiles,
         LATERAL gbx_rst_h3_rastertogridavg(gbx_rst_transform(slope_tile, 4326), {H3_RES}) t
    WHERE t.band = 1
""")

aspect_cells = spark.sql(f"""
    SELECT t.cellID AS cellid, t.measure AS avg_aspect
    FROM sf_terrain_tiles,
         LATERAL gbx_rst_h3_rastertogridavg(gbx_rst_transform(aspect_tile, 4326), {H3_RES}) t
    WHERE t.band = 1
""")

# Inner join on shared H3 cell ids (both rasters cover the same AOI).
cells = slope_cells.join(aspect_cells, on="cellid", how="inner")

# Apply solar_score per cell (from config_nb).
# solar_score(slope_col, aspect_col) expects column NAME strings (not Column objects).
scored = cells.select(
    "cellid",
    "avg_slope",
    "avg_aspect",
    solar_score(slope_col="avg_slope", aspect_col="avg_aspect").alias("solar_score"),
)

# Native H3: h3_centeraswkb requires Photon / Databricks SQL — gate behind try/skip
# so the rest of the notebook stays green where native H3 is absent (e.g., Docker).
_native_h3_ok = False
try:
    scored.selectExpr("h3_centeraswkb(cellid) AS _test").limit(1).collect()
    _native_h3_ok = True
except Exception as _e:
    print(f"... native h3_centeraswkb not available ({type(_e).__name__}); "
          "skipping cell_center_wkb column (requires Photon / Databricks SQL)")

if _native_h3_ok:
    # h3_centeraswkb(cellid BIGINT) -> BINARY (WKB Point at H3 cell center).
    # Confirmed Databricks-native H3 built-in (companions: h3_boundaryaswkb, h3_hexring).
    scored = scored.selectExpr(
        "*",
        "h3_centeraswkb(cellid) AS cell_center_wkb",
    )
    print("... native h3_centeraswkb available — cell_center_wkb column added")

sf_solar_cells = finalize_delta(scored, "sf_solar_cells")
display(sf_solar_cells.orderBy(F.col("solar_score").desc()).limit(10))


## 6. Tile the hillshade to raster PMTiles

Reproject the hillshade to web mercator, pyramid it, and fold into PMTiles — the same
`rst_to_webmercator → gbx_rst_xyzpyramid → gbx_pmtiles_agg` chain as notebook 02, now
over the relief.

Light-tier UDTF output columns (flat): `z INTEGER, x INTEGER, y INTEGER, bytes BINARY`.
Raster PMTiles use first-wins per `(z, x, y)`. One source tile → one non-overlapping
pyramid → no tiles lost.

In [ ]:
# Reproject hillshade tile to web mercator (EPSG:3857) — required for the XYZ grid.
hs_3857 = sf_terrain.select(rx.rst_to_webmercator("hillshade_tile").alias("tile"))
hs_3857.createOrReplaceTempView("sf_hillshade_tile")

# gbx_rst_xyzpyramid (light-tier UDTF) signature:
#   gbx_rst_xyzpyramid(tile, min_z, max_z [, format [, size [, resampling]]])
# Output columns (flat, light tier): z INTEGER, x INTEGER, y INTEGER, bytes BINARY
# Use LATERAL (not LATERAL VIEW) — the UDTF emits rows directly (no explode needed).
hs_xyz = spark.sql("""
    SELECT t.z, t.x, t.y, t.bytes AS png
    FROM sf_hillshade_tile,
         LATERAL gbx_rst_xyzpyramid(tile, 11, 14) t
""")

sf_hs_xyz = finalize_delta(hs_xyz, "sf_hillshade_xyz_tiles")
print(f"... {sf_hs_xyz.count():,} (z,x,y) hillshade tiles across z11-z14")

# Fold into PMTiles (first-wins for raster; one source tile = no overlaps).
# gbx_pmtiles_agg signature: gbx_pmtiles_agg(bytes, z, x, y [, metadata_json])
archive_row = (
    sf_hs_xyz.groupBy(F.lit(1).alias("_g"))
             .agg(F.expr("gbx_pmtiles_agg(png, z, x, y)").alias("archive"))
             .select("archive")
             .collect()[0]
)
TILES_DIR = f"{HELIOS_DIR}/tiles"
os.makedirs(TILES_DIR, exist_ok=True)   # Volume FUSE-safe; idempotent
HS_PMTILES = f"{TILES_DIR}/sf_hillshade.pmtiles"
with open(HS_PMTILES, "wb") as f:
    f.write(archive_row["archive"])
print(f"... wrote {HS_PMTILES} ({os.path.getsize(HS_PMTILES):,} bytes)")

In [ ]:
show_pmtiles(HS_PMTILES)

## 7. Overlay: hillshade + building footprints + solar score

When `INTERACTIVE_PLOTS = True`, `plot_interactive` layers the hillshade PMTiles with
the building footprints from NB01 (both PMTiles archives embedded via MapLibre GL) and
the per-H3-cell solar score as a grid layer. All three layers degrade gracefully when
the upstream archive or table is not yet available.

In [ ]:
# Multi-layer overlay: hillshade PMTiles + NB01 buildings + H3 solar-score grid.
# pmtiles_layer wraps a PMTiles path; grid_layer wraps a Spark DataFrame of H3 cells.
# plot_interactive accepts any mix of layer types and handles budget/fallback.
from databricks.labs.gbx.vizx import pmtiles_layer, grid_layer, plot_interactive as _plot_interactive

BUILDINGS_PMTILES = f"{TILES_DIR}/sf_buildings.pmtiles"

if INTERACTIVE_PLOTS:
    _layers = [pmtiles_layer(HS_PMTILES, label="hillshade")]
    if os.path.exists(BUILDINGS_PMTILES):
        _layers.append(pmtiles_layer(BUILDINGS_PMTILES, label="buildings"))
    # sf_solar_cells: cellid (H3 int), solar_score (double), optionally cell_center_wkb
    if spark.catalog.tableExists("sf_solar_cells"):
        _solar_df = spark.table("sf_solar_cells")
        _layers.append(grid_layer(_solar_df, grid_system="h3", cellid_col="cellid", column="solar_score", label="solar score"))
    _plot_interactive(_layers)
else:
    print("... set INTERACTIVE_PLOTS = True to render the multi-layer MapLibre overlay")

## What we built — and the full picture

- `sf_cog_catalog` (Delta) — the queryable catalog of analysis-ready COGs, keyed by
  Volume path with bounds and CRS metadata.
- `sf_terrain` (Delta) — slope / aspect / hillshade tiles.
- `sf_solar_cells` (Delta) — per-H3-cell avg slope/aspect + `solar_score`, with native
  H3 cell geometry when Photon is available — joins directly with the NB01 roof-density
  cells on the same H3 index.
- `sf_hillshade_xyz_tiles` (Delta) — one row per `(z, x, y)` PNG tile.
- `sf_hillshade.pmtiles` (Volume) — the relief view.

Across the series we built three PMTiles layers over one SF AOI — **buildings**
(vector, NB01), **NAIP aerial** (raster, NB02), and **hillshade** (raster, NB03) —
plus a COG catalog of the elevation. Stack the building footprints over the aerial
basemap, score each roof by the terrain `solar_score`, and you have an end-to-end
distributed solar site-selection pipeline — ingest → tile → PMTiles → view, all on
Databricks.